# mZAR: a small remittance payment platform on our own blockchain

**Financial Blockchain Capstone, African remittances prototype**

This notebook is the whole prototype in one place. A sender pays rand into a simulated bank, receives the same number of **mZAR** coins on our chain, sends them to a recipient, and the recipient redeems them for rand again. Every step leaves a record that anyone can check.

We keep the design close to the two textbooks. The chain, proof of work and Flask API follow van Flymen's *Learn Blockchain by Building One* (Chapters 3 to 7), including its account-based wallets and Ed25519 signatures. The transaction, Merkle tree and verification ideas follow Antonopoulos' *Mastering Bitcoin*, 3rd edition (Chapters 6, 8, 11 and 12).

This is a teaching prototype, not a product. The limitations are listed at the end of the notebook and again in the report.

**How to run it.** Install the packages in `requirements.txt`, then choose *Kernel > Restart and Run All*. The demo uses a fixed clock and fixed demo keys, so every transaction id and block hash comes out the same each time you run it. Only the mining timings change from machine to machine.

| Section | What happens |
|---|---|
| 1 to 3 | Money, wallets and signed transactions |
| 4 to 5 | Blocks, proof of work, Merkle proofs and the ledger rules |
| 6 | Bank, reserve, on-ramp and off-ramp |
| 7 to 8 | The payment API (Flask, bound to localhost) and starting it |
| 9 | Demo script, steps i to vi (steps v and vi sit after the explorer) |
| 10 | Explorer |
| 11 | Extra tests: tampering and proof-of-work timing |
| 12 to 13 | Saved evidence and limitations |

Cross-border remittances remain expensive, particularly in sub-Saharan Africa, where sending US$200 costs approximately 9%, compared to a global average of 6%.

To explore how blockchain technology can improve payment transfers and verification, we developed mZAR, a simulated South African rand-backed stablecoin operating on our own Python-based blockchain.

Our platform has three main components:

Flask API: Allows users to create wallets and submit payment requests.

Blockchain: Records transactions, verifies digital signatures and confirms payments through proof-of-work mining.

Bank and Reserve System: Manages simulated rand deposits and withdrawals, maintaining the rule 1 mZAR = R1.

How the System Works

Users deposit rand into the reserve, receive equivalent mZAR tokens, transfer them through the blockchain and redeem them back into rand.

Our key principle: Rand must enter the reserve before tokens are minted, and tokens must be burned before rand is paid out.

The following sections demonstrate the implementation, payment process, transaction verification and security controls of our prototype.

In [1]:
import copy, hashlib, html, json, logging, os, threading, time
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display, HTML
from flask import Flask, jsonify, request, Response
from nacl.encoding import HexEncoder
from nacl.exceptions import BadSignatureError
from nacl.signing import SigningKey, VerifyKey
from werkzeug.serving import make_server

logging.getLogger("werkzeug").setLevel(logging.ERROR)   # keep the notebook output readable
print("imports ok")

imports ok


### Reliability and security improvements (revised version 2)

**Input validation.** `to_cents` rejects fractional cents, NaN, infinity, zero and negative amounts instead of rounding them. `Ledger.check` type-checks every field, and a `bool` can no longer pass as an integer. Node registration, wallet names and JSON bodies are validated, and API errors now say *why* an amount was refused.

**Mining and concurrency.** `Blockchain.mine` removes from the pool only the transactions it actually mined, and a re-entrant lock stops a payment that arrives *during* proof of work from being lost. A block is applied to a copy of the ledger and swapped in only if every transaction re-validates, so a failure cannot leave a half-updated ledger.

**Reserve integrity.**
- Coins can only be created by `/onramp`, which takes the rand **first** and refunds it if the mint cannot be queued. `/transactions/new` refuses a MINT even if it carries the reserve's valid signature, so a leaked reserve key alone cannot print unbacked coins through the API.
- The wallet name `RESERVE` is forbidden. Without that rule a user could create a wallet called `RESERVE` and on-ramp "from" the reserve's own account for free.
- `settle` marks a redemption as paid before moving the rand and undoes that if the transfer fails, so a crash cannot cause a double payout.
- `/reserve` now reports an exact identity, $R_t = S_t + M_t + U_t$ (pending mints, unsettled redemptions), and checks that every confirmed MINT has a recorded deposit.

**Non-custodial path.** `POST /wallets/register` links a client-held public key to a bank account, so a user who signs on their own device can also redeem. A REDEEM from an unregistered address is refused *before* it can burn coins that could never be paid out.

**Scope.** These changes improve a local teaching prototype. They do not provide durable storage, authenticated API callers, independent consensus, or a truly atomic fiat/blockchain settlement. The demo Reserve key is derived from a name, so anyone who reads this notebook can recompute it; set `MZAR_RESERVE_SEED` to use a private one (hashes then differ from the report). Run the demonstration and negative tests before presenting the results.


## 1. Money and hashing helpers

Money is stored as whole **cents** (integers), never as floats. That way $0.1 + 0.2$ problems cannot creep into a ledger. The helper `to_cents` accepts `1000`, `"1000"` or `"12.50"`.

Everything we hash goes through `canonical`, which writes a dictionary as JSON with sorted keys. Two people hashing the same transaction will always get the same bytes, so they always get the same hash.

In [2]:
def to_cents(amount):
    """Turn a rand amount such as 1000 or '12.50' into whole cents."""
    try:
        value = Decimal(str(amount))
    except (InvalidOperation, ValueError, TypeError):
        raise ValueError(f"not a valid amount: {amount!r}")
    if not value.is_finite() or value <= 0:
        raise ValueError("amount must be a positive finite number")
    cents = value * 100
    if cents != cents.to_integral_value():
        raise ValueError("amount must have no more than two decimal places")
    return int(cents)

def rand(cents):  return f"R{cents / 100:,.2f}"
def mzar(cents):  return f"{cents / 100:,.2f} mZAR"

def sha256(text):
    return hashlib.sha256(text.encode()).hexdigest()

def canonical(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"))

print(to_cents(1000), to_cents("12.50"), rand(125000), mzar(75000))

100000 1250 R1,250.00 750.00 mZAR


## 2. Wallets

A wallet is a key pair. The **private key** signs, the **public key** verifies. As in van Flymen Chapter 6 we use Ed25519 from PyNaCl, and as in his Chapter 7 the hex-encoded public key is simply the account address.

`demo_wallet` builds a key from a name so the demo repeats exactly. **Never do that with real money.** Anyone who knows the name knows the key.

In [3]:
class Wallet:
    can_sign = True                      # the platform holds this key (custodial)
    def __init__(self, name, seed=None):
        self.name = name
        self._key = SigningKey(seed) if seed else SigningKey.generate()
        self.address = self._key.verify_key.encode(encoder=HexEncoder).decode()

    def sign(self, message):
        return self._key.sign(message.encode()).signature.hex()

def demo_wallet(name):
    seed = hashlib.sha256(("mzar-demo-" + name).encode()).digest()
    return Wallet(name, seed=seed)

def verify_signature(address, message, signature_hex):
    try:
        VerifyKey(address, encoder=HexEncoder).verify(message.encode(), bytes.fromhex(signature_hex))
        return True
    except (BadSignatureError, ValueError, TypeError):
        return False

w = demo_wallet("Example")
sig = w.sign("hello")
print("address :", w.address)
print("good sig:", verify_signature(w.address, "hello", sig))
print("bad sig :", verify_signature(w.address, "hello!", sig))

address : a5bf48c2d6d0c48bd3e8fff517830046ac7ccea1ccc67f07d55fbaba2ca847ff
good sig: True
bad sig : False


## 3. Transactions

Only three kinds of transaction exist.

| Kind | Who signs | Effect |
|---|---|---|
| `MINT` | the reserve agent | creates new mZAR for a user (the on-ramp) |
| `TRANSFER` | the sender | moves mZAR between two users |
| `REDEEM` | the holder | burns mZAR and asks the reserve to pay out rand (the off-ramp) |

Each transaction carries a **nonce**, which is the sender's own running count of transactions (0, 1, 2, ...). This is the account-model way to stop replays, the same idea Ethereum uses. The id is the hash of the signed fields.

In [4]:
FIELDS = ("kind", "sender", "recipient", "amount", "nonce")

def tx_body(tx):
    return {k: tx[k] for k in FIELDS}

def make_tx(wallet, kind, recipient, amount, nonce):
    body = {"kind": kind, "sender": wallet.address, "recipient": recipient,
            "amount": amount, "nonce": nonce}
    message = canonical(body)
    return {**body, "id": sha256(message), "signature": wallet.sign(message)}

## 4. Blocks, proof of work and Merkle proofs

A block header holds the index, a timestamp, the previous block's hash, the **Merkle root** of its transactions, the difficulty and a nonce. Mining means trying nonces until the header hash starts with `difficulty` zeros (van Flymen Chapter 4, which Bitcoin calls Hashcash).

With a hex hash, each leading zero has probability $1/16$, so a difficulty of $d$ needs about $16^d$ attempts on average. We use $d = 4$ for the demo, which is roughly 65,000 attempts and finishes in a fraction of a second.

The Merkle root lets anyone prove that one transaction sits in a block by showing only a short path of hashes, without downloading every transaction (Mastering Bitcoin, Chapters 10 and 11, on simplified payment verification).

In [5]:
def merkle_root(ids):
    if not ids:
        return sha256("")
    level = list(ids)
    while len(level) > 1:
        if len(level) % 2:
            level.append(level[-1])
        level = [sha256(level[i] + level[i + 1]) for i in range(0, len(level), 2)]
    return level[0]

def merkle_proof(ids, index):
    """The sibling hashes needed to walk from one transaction id up to the root."""
    proof, level, i = [], list(ids), index
    while len(level) > 1:
        if len(level) % 2:
            level.append(level[-1])
        sibling = i ^ 1
        proof.append([level[sibling], "left" if sibling < i else "right"])
        level = [sha256(level[j] + level[j + 1]) for j in range(0, len(level), 2)]
        i //= 2
    return proof

def verify_proof(tx_id, proof, root):
    h = tx_id
    for sibling, side in proof:
        h = sha256(sibling + h) if side == "left" else sha256(h + sibling)
    return h == root

def header_hash(b):
    return sha256(f"{b['index']}|{b['timestamp']}|{b['previous_hash']}|"
                  f"{b['merkle_root']}|{b['difficulty']}|{b['nonce']}")

# quick self-test with five fake ids
ids = [sha256(str(n)) for n in range(5)]
root = merkle_root(ids)
print("proof for tx 3 valid:", verify_proof(ids[3], merkle_proof(ids, 3), root))
print("wrong tx rejected   :", not verify_proof(ids[2], merkle_proof(ids, 3), root))

proof for tx 3 valid: True
wrong tx rejected   : True


## 5. The ledger rules and the blockchain

`Ledger` is the account table: balance and nonce for every address, plus total supply. Its `check` method holds **all** the rules in one place, so the same code protects the pending pool, mining, and the full chain audit.

The `Blockchain` class is the van Flymen chain with three changes: it uses signed account transactions, it stores a Merkle root in each header, and `verify_chain` replays history from genesis through the same `check` rules.

There is no miner reward and no fee. In this prototype a single platform operator mines. Section 13 and the report explain why that is a simplification.

In [6]:
class Ledger:
    """Balance and nonce of every address, plus total supply of mZAR."""
    def __init__(self, reserve):
        self.reserve = reserve
        self.balances, self.nonces, self.supply = {}, {}, 0

    def copy(self):
        return copy.deepcopy(self)

    def balance(self, address):
        return self.balances.get(address, 0)

    def check(self, tx):
        """None if the transaction is valid, otherwise a short reason."""
        if not isinstance(tx, dict) or any(k not in tx for k in FIELDS + ("id", "signature")):
            return "malformed transaction"
        if not all(isinstance(tx[k], str) for k in ("kind", "sender", "recipient", "id", "signature")):
            return "transaction identifiers and signature must be strings"
        if not isinstance(tx["nonce"], int) or isinstance(tx["nonce"], bool) or tx["nonce"] < 0:
            return "nonce must be a non-negative integer"
        if tx["kind"] not in ("MINT", "TRANSFER", "REDEEM"):
            return "unknown transaction kind"
        if not isinstance(tx["amount"], int) or isinstance(tx["amount"], bool) or tx["amount"] <= 0:
            return "amount must be a positive whole number of cents"
        try:
            signature_ok = verify_signature(tx["sender"], canonical(tx_body(tx)), tx["signature"])
        except (ValueError, TypeError, KeyError):
            signature_ok = False
        if not signature_ok:
            return "invalid signature"
        if tx["id"] != sha256(canonical(tx_body(tx))):
            return "transaction id does not match its contents"
        expected = self.nonces.get(tx["sender"], 0)
        if tx["nonce"] != expected:
            return f"bad nonce (expected {expected}, got {tx['nonce']})"
        if tx["kind"] == "MINT":
            if tx["sender"] != self.reserve:
                return "only the reserve agent can mint"
        elif self.balance(tx["sender"]) < tx["amount"]:
            return "insufficient balance"
        if tx["kind"] == "TRANSFER":
            if tx["recipient"] == tx["sender"]:
                return "cannot pay yourself"
            if tx["recipient"] == self.reserve:
                return "pay the reserve with a REDEEM, not a TRANSFER"
        if tx["kind"] == "REDEEM" and tx["recipient"] != self.reserve:
            return "a REDEEM must be addressed to the reserve"
        return None

    def apply(self, tx):
        kind, s, r, a = tx["kind"], tx["sender"], tx["recipient"], tx["amount"]
        if kind == "MINT":
            self.balances[r] = self.balance(r) + a
            self.supply += a
        elif kind == "TRANSFER":
            self.balances[s] = self.balance(s) - a
            self.balances[r] = self.balance(r) + a
        else:                                   # REDEEM burns the coins
            self.balances[s] = self.balance(s) - a
            self.supply -= a
        self.nonces[s] = self.nonces.get(s, 0) + 1


class Blockchain:
    def __init__(self, reserve, difficulty=4, clock=time.time):
        self.reserve, self.difficulty, self.clock = reserve, difficulty, clock
        self.chain, self.pending, self.nodes = [], [], set()
        self.ledger = Ledger(reserve)
        self.tx_location = {}                   # tx id -> (block number, position)
        self.lock = threading.RLock()           # one writer at a time (the API server is threaded)
        self._add_block([])                     # genesis

    def __deepcopy__(self, memo):
        """Copy everything except the lock, which cannot be copied."""
        clone = self.__class__.__new__(self.__class__)
        for key, value in self.__dict__.items():
            clone.__dict__[key] = threading.RLock() if key == "lock" else copy.deepcopy(value, memo)
        return clone

    def _mine(self, txs):
        prev = self.chain[-1]["hash"] if self.chain else "0" * 64
        block = {"index": len(self.chain), "timestamp": self.clock(),
                 "previous_hash": prev,
                 "merkle_root": merkle_root([t["id"] for t in txs]),
                 "difficulty": self.difficulty, "nonce": 0}
        target = "0" * self.difficulty
        while not header_hash(block).startswith(target):
            block["nonce"] += 1
        block["hash"] = header_hash(block)
        block["transactions"] = txs
        return block

    def _add_block(self, txs):
        """Mine a block and apply it. If anything fails, the live state is left untouched."""
        with self.lock:
            block = self._mine(txs)
            ledger = self.ledger.copy()         # apply to a copy first ...
            for tx in txs:
                reason = ledger.check(tx)       # ... re-validating each transaction in order
                if reason:
                    raise ValueError(f"cannot mine: {reason}")
                ledger.apply(tx)
            self.ledger = ledger                # ... then swap it in
            for pos, tx in enumerate(txs):
                self.tx_location[tx["id"]] = (block["index"], pos)
            self.chain.append(block)
            return block

    def projected(self):
        """The ledger as it will look once the pending transactions are mined."""
        with self.lock:
            ledger = self.ledger.copy()
            for tx in self.pending:
                ledger.apply(tx)
            return ledger

    def next_nonce(self, address):
        return self.projected().nonces.get(address, 0)

    def submit(self, tx):
        with self.lock:
            reason = self.projected().check(tx)
            if reason:
                return False, reason
            self.pending.append(tx)
            return True, "accepted into the pending pool"

    def mine(self):
        with self.lock:                         # payments that arrive during proof of work wait here
            if not self.pending:
                raise ValueError("nothing to mine")
            txs = list(self.pending)
            block = self._add_block(txs)        # raises before touching the pool if it fails
            del self.pending[:len(txs)]         # drop only what was actually mined
            return block

    def lookup(self, tx_id):
        with self.lock:
            if tx_id in self.tx_location:
                n, pos = self.tx_location[tx_id]
                block = self.chain[n]
                ids = [t["id"] for t in block["transactions"]]
                proof = merkle_proof(ids, pos)
                return {"status": "confirmed", "block": n, "block_hash": block["hash"],
                        "position": pos, "confirmations": len(self.chain) - n,
                        "tx": block["transactions"][pos], "merkle_root": block["merkle_root"],
                        "merkle_proof": proof,
                        "proof_valid": verify_proof(tx_id, proof, block["merkle_root"])}
            for tx in self.pending:
                if tx["id"] == tx_id:
                    return {"status": "pending", "tx": tx}
            return None

    def verify_chain(self):
        """Replay the whole chain from genesis. Returns (ok, message)."""
        with self.lock:
            ledger, prev, last_time, seen = Ledger(self.reserve), "0" * 64, None, set()
            for i, b in enumerate(self.chain):
                if b["index"] != i:
                    return False, f"block {i}: wrong index"
                if b["previous_hash"] != prev:
                    return False, f"block {i}: previous_hash does not match block {i - 1}"
                if b["hash"] != header_hash(b):
                    return False, f"block {i}: stored hash does not match the header"
                if b["difficulty"] < self.difficulty or not b["hash"].startswith("0" * b["difficulty"]):
                    return False, f"block {i}: proof of work missing"
                if last_time is not None and b["timestamp"] < last_time:
                    return False, f"block {i}: timestamp goes backwards"
                if b["merkle_root"] != merkle_root([t["id"] for t in b["transactions"]]):
                    return False, f"block {i}: merkle root does not match its transactions"
                for tx in b["transactions"]:
                    if tx.get("id") in seen:    # also closes the odd-leaf duplication trick in Merkle trees
                        return False, f"block {i}: transaction {tx['id'][:10]} appears twice"
                    seen.add(tx.get("id"))
                    reason = ledger.check(tx)
                    if reason:
                        return False, f"block {i}, tx {tx['id'][:10]}: {reason}"
                    ledger.apply(tx)
                prev, last_time = b["hash"], b["timestamp"]
            return True, f"chain is valid ({len(self.chain)} blocks, supply {mzar(ledger.supply)})"


## 6. Bank, reserve, on-ramp and off-ramp

The chain only knows about mZAR. The rand lives in a toy `Bank`, which stands in for a real bank account. The `PaymentService` is the glue, and it plays the **reserve agent**.

**The reserve rule is 1:1.** Writing $S_t$ for mZAR in circulation and $R_t$ for rand held in the reserve account:

$$\rho_t = \frac{R_t}{S_t} \ge 1$$

The ratio equals 1 whenever nothing is in flight. It is briefly above 1 while a deposit waits to be mined, or while a burned redemption waits for payout. Both are safe directions, because the reserve never owes more than it holds.

The books actually satisfy an exact identity at every moment: $R_t = S_t + M_t + U_t$, where $M_t$ is the amount of MINT transactions still pending (rand received, coins not yet minted) and $U_t$ the amount of confirmed REDEEMs not yet paid out. `GET /reserve` reports it as `reserve_identity_holds`, and `mints_match_deposits` checks that every confirmed MINT on the chain has a recorded deposit. This is an *off-chain* check run by the reserve agent: the chain itself cannot see the bank, which is why a real reserve would also need an outside attestation.

- **On-ramp:** the user pays rand to the reserve **first**, then the reserve signs a `MINT` (if the mint cannot be queued, the rand is refunded). The conversion is recorded twice: as a chain transaction, and as an event in the service log.
- **Off-ramp, two steps:** the user signs a `REDEEM`, which burns the coins once mined. Only after the reserve sees that transaction **confirmed in a block, with a valid Merkle proof**, does it pay rand out and write a receipt. A transaction can only be settled once.

In [7]:
class PaymentError(Exception):
    pass

class Bank:
    """A toy ZAR bank ledger, in cents."""
    def __init__(self):
        self.accounts = {}
    def open(self, name, cents=0):
        self.accounts[name] = cents
    def move(self, src, dst, cents):
        if not isinstance(cents, int) or isinstance(cents, bool) or cents <= 0:
            raise PaymentError("bank transfer must be a positive number of cents")
        if src == dst:
            raise PaymentError("cannot move rand to the same account")
        if self.accounts.get(src, 0) < cents:
            raise PaymentError(f"{src} does not have enough rand")
        self.accounts[src] -= cents
        self.accounts[dst] = self.accounts.get(dst, 0) + cents


class ExternalWallet:
    """A wallet whose private key the platform never sees (non-custodial)."""
    can_sign = False
    def __init__(self, name, address):
        self.name, self.address = name, address
    def sign(self, message):
        raise PaymentError(f"{self.name} is a non-custodial wallet: sign on your own device "
                           f"and POST the transaction to /transactions/new")


class PaymentService:
    def __init__(self, chain, reserve_wallet, bank, make_wallet=Wallet):
        self.chain, self.reserve, self.bank = chain, reserve_wallet, bank
        self.make_wallet = make_wallet
        self.wallets, self.events, self.receipts, self.settled = {}, [], [], set()
        self.lock = chain.lock                  # share the chain's lock: payments and mining never interleave
        self.bank.open("RESERVE")

    def add_wallet(self, wallet):
        with self.lock:
            if wallet.name.strip().upper() == "RESERVE":
                raise PaymentError("the name RESERVE is reserved for the reserve account")
            self.wallets[wallet.name] = wallet
            self.bank.accounts.setdefault(wallet.name, 0)
            return wallet

    def register_external(self, name, address):
        """Link a client-held public key to a bank account (non-custodial path)."""
        with self.lock:
            if not isinstance(name, str) or not name.strip():
                raise PaymentError("name must be a non-empty string")
            if name in self.wallets:
                raise PaymentError("a wallet with that name already exists")
            if (not isinstance(address, str) or len(address) != 64
                    or any(c not in "0123456789abcdef" for c in address)):
                raise PaymentError("address must be 64 lowercase hex characters")
            if address == self.reserve.address or self._registered(address):
                raise PaymentError("that address is already registered")
            return self.add_wallet(ExternalWallet(name, address))

    def _registered(self, address):
        return any(w.address == address for w in self.wallets.values())

    def _wallet(self, name):
        if not isinstance(name, str) or name not in self.wallets:
            raise PaymentError(f"no wallet called {name!r}")
        return self.wallets[name]

    def _submit(self, wallet, kind, recipient, cents):
        if cents <= 0:
            raise PaymentError("amount must be positive")
        tx = make_tx(wallet, kind, recipient, cents, self.chain.next_nonce(wallet.address))
        ok, why = self.chain.submit(tx)
        if not ok:
            raise PaymentError(why)
        return tx

    def onramp(self, name, cents):
        """Rand first, coins second. If the mint cannot be queued, the rand is refunded."""
        with self.lock:
            w = self._wallet(name)
            if cents <= 0:
                raise PaymentError("amount must be positive")
            if self.bank.accounts.get(name, 0) < cents:
                raise PaymentError(f"{name} does not have enough rand in the bank")
            self.bank.move(name, "RESERVE", cents)
            try:
                tx = self._submit(self.reserve, "MINT", w.address, cents)
            except Exception:
                self.bank.move("RESERVE", name, cents)
                raise
            event = {"event": "ONRAMP", "who": name, "rand_in": cents, "mzar_out": cents,
                     "rule": "1 ZAR : 1 mZAR", "tx_id": tx["id"]}
            self.events.append(event)
            return tx, event

    def send(self, frm, to, cents):
        with self.lock:
            return self._submit(self._wallet(frm), "TRANSFER", self._wallet(to).address, cents)

    def request_redeem(self, name, cents):
        with self.lock:
            return self._submit(self._wallet(name), "REDEEM", self.reserve.address, cents)

    def submit_external(self, tx):
        """The public door for transactions signed elsewhere (POST /transactions/new)."""
        with self.lock:
            kind = tx.get("kind") if isinstance(tx, dict) else None
            if kind == "REDEEM" and not self._registered(tx.get("sender")):
                return False, "redeem sender has no registered bank account (POST /wallets/register first)"
            if kind == "MINT":
                reason = self.chain.projected().check(tx)
                if reason:
                    return False, reason
                return False, "mints are created only by the on-ramp, which takes the rand first"
            return self.chain.submit(tx)

    def settle(self, tx_id):
        if not isinstance(tx_id, str):
            raise PaymentError("tx_id must be a string")
        with self.lock:
            if tx_id in self.settled:
                raise PaymentError("this redemption was already paid out")
            found = self.chain.lookup(tx_id)
            if not found or found["status"] != "confirmed":
                raise PaymentError("redeem transaction is not in a block yet")
            tx = found["tx"]
            if tx["kind"] != "REDEEM" or not found["proof_valid"]:
                raise PaymentError("not a valid, confirmed redeem transaction")
            name = next((n for n, w in self.wallets.items() if w.address == tx["sender"]), None)
            if name is None:
                raise PaymentError("redemption sender is not a registered wallet")
            if self.bank.accounts.get("RESERVE", 0) < tx["amount"]:
                raise PaymentError("reserve does not have enough ZAR for settlement")
            self.settled.add(tx_id)             # claim the payout first ...
            try:
                body = {"receipt_no": f"RCPT-{len(self.receipts) + 1:04d}", "holder": name,
                        "mzar_burned": tx["amount"], "rand_paid": tx["amount"],
                        "redeem_tx": tx_id, "block": found["block"], "block_hash": found["block_hash"],
                        "paid_at": self.chain.clock()}
                receipt = {**body, "receipt_hash": sha256(canonical(body))}
                self.bank.move("RESERVE", name, tx["amount"])
            except Exception:
                self.settled.discard(tx_id)     # ... and give it back if the payout failed
                raise
            self.receipts.append(receipt)
            return receipt

    def reserve_status(self):
        with self.lock:
            supply, held = self.chain.ledger.supply, self.bank.accounts["RESERVE"]
            confirmed = [t for b in self.chain.chain for t in b["transactions"]]
            pending_mints = sum(t["amount"] for t in self.chain.pending if t["kind"] == "MINT")
            pending_redemptions = sum(t["amount"] for t in self.chain.pending if t["kind"] == "REDEEM")
            unsettled = sum(t["amount"] for t in confirmed
                            if t["kind"] == "REDEEM" and t["id"] not in self.settled)
            minted = sum(t["amount"] for t in confirmed if t["kind"] == "MINT")
            deposits = sum(e["rand_in"] for e in self.events if e["tx_id"] in self.chain.tx_location)
            return {"mzar_in_circulation": supply, "rand_held_in_reserve": held,
                    "pending_mints": pending_mints,
                    "pending_redemptions": pending_redemptions,
                    "unsettled_redemptions": unsettled,
                    "reserve_identity_holds": held == supply + pending_mints + unsettled,
                    "mints_match_deposits": minted == deposits,
                    "reserve_covers_confirmed_and_pending_mints": held >= supply + pending_mints,
                    "fully_backed": held >= supply,
                    "backing_ratio": round(held / supply, 4) if supply else None,
                    "balances_add_up": sum(self.chain.ledger.balances.values()) == supply}


## 7. The payment API

A small Flask app, in the style of van Flymen Chapter 7. It keeps the endpoints from our earlier API assignment (`/chain`, `/transactions/new`, `/mine`, `/nodes`, `/nodes/register`) and adds the payment flow on top.

| Method and path | Purpose |
|---|---|
| `POST /wallets/new` | create a wallet (returns the address only) |
| `POST /wallets/register` | link a client-held public key to a bank account (non-custodial) |
| `POST /onramp` | pay rand in, mint mZAR (pending until mined) |
| `POST /payments/send` | build, sign and submit a transfer for a demo wallet |
| `POST /transactions/new` | submit a transaction that was signed elsewhere |
| `POST /mine` | mine the pending pool into a block |
| `POST /offramp` | submit a redeem request |
| `POST /offramp/settle` | after confirmation, pay out rand and issue a receipt |
| `GET /transactions/<id>` | look up a transaction with its Merkle proof |
| `GET /chain`, `GET /chain/verify` | the full chain, and a full audit |
| `GET /balances`, `GET /reserve` | balances and the reserve check |
| `GET /explorer` | human-readable HTML view |
| `GET /nodes`, `POST /nodes/register` | peer list (no consensus in this prototype) |

`/transactions/new` refuses every `MINT`, even one carrying the reserve's valid signature: coins are created only by `/onramp`, which takes the rand first. It also refuses a `REDEEM` from an address with no registered bank account, because such a burn could never be paid out. The server binds to `127.0.0.1` only. `/payments/send` signs with keys held by the service, which is **custodial** and only acceptable for a demo. A real client would sign on its own device and call `/transactions/new`.

In [8]:
CSS = ("body{font-family:Georgia,serif;margin:2rem;max-width:62rem}"
       "table{border-collapse:collapse;width:100%;margin-bottom:1.2rem}"
       "td,th{border:1px solid #888;padding:4px 8px;font-size:0.85rem;text-align:left}"
       "code{font-size:0.78rem;word-break:break-all}")

def label(service, address):
    if address == service.reserve.address:
        return "Reserve"
    for name, w in service.wallets.items():
        if w.address == address:
            return name
    return address[:10] + "..."

def explorer_html(service, highlight=None):
    chain = service.chain
    out = [f"<html><head><meta charset='utf-8'><title>mZAR explorer</title><style>{CSS}</style></head><body>",
           f"<h1>mZAR explorer</h1><p>{len(chain.chain)} blocks, difficulty {chain.difficulty}, "
           f"{len(chain.pending)} pending transactions</p>"]
    for b in reversed(chain.chain):
        out.append(f"<h2>Block {b['index']}</h2><p>hash <code>{b['hash']}</code><br>"
                   f"previous <code>{b['previous_hash']}</code><br>"
                   f"nonce {b['nonce']}, merkle root <code>{b['merkle_root']}</code></p>")
        if b["transactions"]:
            out.append("<table><tr><th>Transaction id</th><th>Kind</th><th>From</th><th>To</th><th>mZAR</th></tr>")
            for t in b["transactions"]:
                mark = " <b>&larr; this payment</b>" if t["id"] == highlight else ""
                out.append(f"<tr><td><code>{t['id']}</code>{mark}</td><td>{t['kind']}</td>"
                           f"<td>{html.escape(label(service, t['sender']))}</td>"
                           f"<td>{html.escape(label(service, t['recipient']))}</td>"
                           f"<td>{t['amount'] / 100:,.2f}</td></tr>")
            out.append("</table>")
    out.append("</body></html>")
    return "".join(out)

def create_app(service):
    app, chain = Flask("mzar"), service.chain

    def fail(message, code=400):
        return jsonify({"error": message}), code

    def body():
        data = request.get_json(silent=True)
        return data if isinstance(data, dict) else {}

    def amount_from(data, key):
        try:
            return to_cents(data[key])
        except KeyError:
            raise PaymentError(f"{key} is required")
        except ValueError as e:
            raise PaymentError(f"{key}: {e}")

    @app.errorhandler(PaymentError)
    def payment_error(e):
        return fail(str(e))

    @app.post("/wallets/new")
    def wallets_new():
        name = body().get("name")
        if not isinstance(name, str) or not name.strip():
            return fail("name must be a non-empty string")
        if name in service.wallets:
            return fail("a wallet with that name already exists")
        w = service.add_wallet(service.make_wallet(name))
        return jsonify({"name": name, "address": w.address}), 201

    @app.post("/wallets/register")
    def wallets_register():
        d = body()
        w = service.register_external(d.get("name"), d.get("address"))
        return jsonify({"name": w.name, "address": w.address, "custody": "client holds the key"}), 201

    @app.post("/onramp")
    def onramp():
        d = body()
        tx, event = service.onramp(d.get("name"), amount_from(d, "amount_zar"))
        return jsonify({"tx": tx, "event": event, "note": "mint is pending until the next block"}), 201

    @app.post("/payments/send")
    def payments_send():
        d = body()
        tx = service.send(d.get("from"), d.get("to"), amount_from(d, "amount"))
        return jsonify({"tx": tx, "note": "pending until the next block"}), 201

    @app.post("/transactions/new")
    def transactions_new():
        tx = body()
        ok, why = service.submit_external(tx)
        if not ok:
            return fail(why)
        return jsonify({"message": why, "tx_id": tx["id"]}), 201

    @app.post("/mine")
    def mine():
        try:
            b = chain.mine()
        except ValueError as e:
            return fail(str(e))
        return jsonify({"index": b["index"], "hash": b["hash"], "nonce": b["nonce"],
                        "difficulty": b["difficulty"],
                        "tx_ids": [t["id"] for t in b["transactions"]]}), 201

    @app.post("/offramp")
    def offramp():
        d = body()
        tx = service.request_redeem(d.get("name"), amount_from(d, "amount"))
        return jsonify({"tx": tx, "note": "burn is pending until the next block"}), 201

    @app.post("/offramp/settle")
    def offramp_settle():
        return jsonify(service.settle(body().get("tx_id"))), 201

    @app.get("/transactions/<tx_id>")
    def transaction(tx_id):
        found = chain.lookup(tx_id)
        return (jsonify(found), 200) if found else fail("unknown transaction id", 404)

    @app.get("/chain")
    def full_chain():
        return jsonify({"length": len(chain.chain), "chain": chain.chain})

    @app.get("/chain/verify")
    def chain_verify():
        ok, message = chain.verify_chain()
        return jsonify({"valid": ok, "message": message, "blocks": len(chain.chain)})

    @app.get("/balances")
    def balances():
        rows = {n: {"mzar": chain.ledger.balance(w.address), "bank_zar": service.bank.accounts.get(n, 0)}
                for n, w in service.wallets.items()}
        rows["RESERVE"] = {"mzar": 0, "bank_zar": service.bank.accounts["RESERVE"]}
        return jsonify(rows)

    @app.get("/reserve")
    def reserve():
        return jsonify(service.reserve_status())

    @app.get("/explorer")
    def explorer():
        return Response(explorer_html(service, request.args.get("tx")), mimetype="text/html")

    @app.get("/nodes")
    def nodes():
        return jsonify({"nodes": sorted(chain.nodes)})

    @app.post("/nodes/register")
    def nodes_register():
        listed = body().get("nodes")
        if not isinstance(listed, list) or not listed or not all(isinstance(x, str) and x.strip() for x in listed):
            return fail("nodes must be a non-empty list of addresses")
        chain.nodes.update(listed)
        return jsonify({"message": "nodes added", "nodes": sorted(chain.nodes)}), 201

    return app


class Server(threading.Thread):
    def __init__(self, app, port):
        super().__init__(daemon=True)
        self.httpd = make_server("127.0.0.1", port, app, threaded=True)
    def run(self):
        self.httpd.serve_forever()
    def stop(self):
        self.httpd.shutdown()
        self.httpd.server_close()

## 8. Start the platform

We build the chain with a fixed demo clock (one minute per block), create the reserve agent's wallet, and start the API on `127.0.0.1:5055`. Wallet keys are derived from names only because this is a demo.

In [9]:
class DemoClock:
    """Moves one minute forward each time it is read, so the demo is repeatable."""
    def __init__(self, start=1791705600, step=60):
        self.t, self.step = start - step, step
    def __call__(self):
        self.t += self.step
        return self.t

DIFFICULTY = 4
PORT = 5055
BASE = f"http://127.0.0.1:{PORT}"

if "server" in globals():
    server.stop()                      # lets us re-run this cell safely

_seed = os.environ.get("MZAR_RESERVE_SEED")
if _seed:
    reserve_wallet = Wallet("Reserve", seed=hashlib.sha256(("mzar-reserve-" + _seed).encode()).digest())
    print("reserve key: private, from MZAR_RESERVE_SEED (hashes will differ from the report)")
else:
    reserve_wallet = demo_wallet("Reserve")
    print("reserve key: DEMO key derived from the name 'Reserve'. Anyone can recompute it. Demo only.")
chain = Blockchain(reserve_wallet.address, difficulty=DIFFICULTY, clock=DemoClock())
bank = Bank()
service = PaymentService(chain, reserve_wallet, bank, make_wallet=demo_wallet)
server = Server(create_app(service), PORT)
server.start()

print("API running at", BASE)
print("genesis block hash:", chain.chain[0]["hash"])
print("reserve address   :", reserve_wallet.address)

reserve key: DEMO key derived from the name 'Reserve'. Anyone can recompute it. Demo only.
API running at http://127.0.0.1:5055
genesis block hash: 0000adebcfbaf1d8fc5075328333e3f6e6a3f612838fb15d905a2552cb397d18
reserve address   : 8911be18f32f00dcdf7e949f701e116de484be67a8504e1f0896c23769548e1f


In [10]:
def call(method, path, **payload):
    r = requests.request(method, BASE + path, json=payload or None, timeout=60)
    return r.status_code, r.json()

def show(title, status, data, keys=None):
    shown = {k: data[k] for k in keys if k in data} if keys else data
    print(f"{title}\n  HTTP {status}")
    for k, v in shown.items():
        print(f"  {k}: {v}")
    print()

def money_table():
    _, rows = call("GET", "/balances")
    df = pd.DataFrame({n: {"mZAR on chain": mzar(v["mzar"]), "ZAR in bank": rand(v["bank_zar"])}
                       for n, v in rows.items()}).T
    return df

def reserve_line():
    _, r = call("GET", "/reserve")
    print(f"reserve: {mzar(r['mzar_in_circulation'])} in circulation, {rand(r['rand_held_in_reserve'])} held, "
          f"ratio {r['backing_ratio']}, fully backed: {r['fully_backed']}, balances add up: {r['balances_add_up']}, "
          f"identity holds: {r['reserve_identity_holds']}, mints match deposits: {r['mints_match_deposits']}")
    return r

evidence = {"difficulty": DIFFICULTY, "api": BASE}

## 9. Demo script

The brief asks for six steps: wallets, on-ramp, transfer, off-ramp, verification and a failed transfer. Each one below calls the API over HTTP, exactly as a client would.

### Step i. Create wallets

Sipho works in Johannesburg and wants to send money home. Lindiwe is his recipient. The reserve agent already exists. To make the on-ramp meaningful we give Sipho R5,000.00 in his (simulated) bank account.

In [11]:
for name in ["Sipho", "Lindiwe"]:
    status, data = call("POST", "/wallets/new", name=name)
    show(f"create wallet {name}", status, data)

bank.open("Sipho", to_cents(5000))        # pre-existing bank balance, set outside the API
money_table()

create wallet Sipho
  HTTP 201
  address: 6b88d232fdc1c94cd8a3985876f1dfe9d1327e325014cdc6cf16baa920f25915
  name: Sipho

create wallet Lindiwe
  HTTP 201
  address: 66e39706e8d015d3c59c2eb13ecbbb00d106a3e96ecd43a70a18917fff5b0aed
  name: Lindiwe



,mZAR on chain,ZAR in bank
Lindiwe,0.00 mZAR,R0.00
RESERVE,0.00 mZAR,R0.00
Sipho,0.00 mZAR,"R5,000.00"


### Step ii. On-ramp: R1,000.00 becomes 1,000.00 mZAR

The deposit creates a pending `MINT`. We mine a block so it becomes part of the chain, then check the balances and the reserve.

In [12]:
status, data = call("POST", "/onramp", name="Sipho", amount_zar=1000)
show("on-ramp R1,000.00", status, data["event"])
evidence["onramp_tx"] = data["tx"]["id"]

status, blk = call("POST", "/mine")
show("mine block", status, blk, keys=["index", "hash", "nonce", "difficulty"])
evidence["block_onramp"] = {"index": blk["index"], "hash": blk["hash"], "nonce": blk["nonce"]}

display(money_table())
evidence["reserve_after_onramp"] = reserve_line()

on-ramp R1,000.00
  HTTP 201
  event: ONRAMP
  mzar_out: 100000
  rand_in: 100000
  rule: 1 ZAR : 1 mZAR
  tx_id: 9f4b12c94b30b0db142baa799a0b79c76b1034504ebbd777ae256e98e6618491
  who: Sipho

mine block
  HTTP 201
  index: 1
  hash: 000063e3b385282412005502e64d339f3d5b502f20e04684534ece541091ad29
  nonce: 7663
  difficulty: 4



,mZAR on chain,ZAR in bank
Lindiwe,0.00 mZAR,R0.00
RESERVE,0.00 mZAR,"R1,000.00"
Sipho,"1,000.00 mZAR","R4,000.00"


reserve: 1,000.00 mZAR in circulation, R1,000.00 held, ratio 1.0, fully backed: True, balances add up: True, identity holds: True, mints match deposits: True


### Step iii. Transfer: Sipho sends 750.00 mZAR to Lindiwe

This is the signed stablecoin transfer. It enters the pending pool, then goes into a mined block.

In [13]:
status, data = call("POST", "/payments/send", **{"from": "Sipho", "to": "Lindiwe", "amount": 750})
transfer_tx = data["tx"]
show("send 750 mZAR", status, transfer_tx, keys=["id", "kind", "amount", "nonce", "signature"])
evidence["transfer_tx"] = transfer_tx["id"]

status, blk = call("POST", "/mine")
show("mine block", status, blk, keys=["index", "hash", "nonce", "difficulty"])
evidence["block_transfer"] = {"index": blk["index"], "hash": blk["hash"], "nonce": blk["nonce"]}

display(money_table())
evidence["reserve_after_transfer"] = reserve_line()

send 750 mZAR
  HTTP 201
  id: 49ecdae540d323efd9e724a2925903a167f10d5ff49a171b52cfd708193a57d7
  kind: TRANSFER
  amount: 75000
  nonce: 0
  signature: 3f09cf93dc8fa39c7b0b0b2c82ce87f18910af7fd6b285c49943fb96475886e250e2f88243e44309bad9883c86c059ecb76420e514e982d82b3be6a75c078a01

mine block
  HTTP 201
  index: 2
  hash: 00003bde11490d5d0f839a139948070d3b2e4ed06e86b921adf7d1b885101a32
  nonce: 59849
  difficulty: 4



,mZAR on chain,ZAR in bank
Lindiwe,750.00 mZAR,R0.00
RESERVE,0.00 mZAR,"R1,000.00"
Sipho,250.00 mZAR,"R4,000.00"


reserve: 1,000.00 mZAR in circulation, R1,000.00 held, ratio 1.0, fully backed: True, balances add up: True, identity holds: True, mints match deposits: True


### Step iv. Off-ramp: Lindiwe redeems 750.00 mZAR

The redeem request is mined first. The reserve then looks the transaction up, checks its Merkle proof, pays out rand and issues a receipt.

In [14]:
status, data = call("POST", "/offramp", name="Lindiwe", amount=750)
redeem_tx = data["tx"]
show("redeem request", status, redeem_tx, keys=["id", "kind", "amount", "nonce"])
evidence["redeem_tx"] = redeem_tx["id"]

# settling before the block exists must fail
status, data = call("POST", "/offramp/settle", tx_id=redeem_tx["id"])
show("settle too early (should be refused)", status, data)

status, blk = call("POST", "/mine")
show("mine block", status, blk, keys=["index", "hash", "nonce", "difficulty"])
evidence["block_redeem"] = {"index": blk["index"], "hash": blk["hash"], "nonce": blk["nonce"]}

status, receipt = call("POST", "/offramp/settle", tx_id=redeem_tx["id"])
show("settle after confirmation", status, receipt)
evidence["receipt"] = receipt

display(money_table())
evidence["reserve_after_offramp"] = reserve_line()

redeem request
  HTTP 201
  id: 5270433f2cc0072f76c63d8e817b88364779b0c08891936b93fe63fd44e0e67a
  kind: REDEEM
  amount: 75000
  nonce: 0

settle too early (should be refused)
  HTTP 400
  error: redeem transaction is not in a block yet

mine block
  HTTP 201
  index: 3
  hash: 000024325f90e44bd629de3ae9214952528ab87c0ac68d9bbdc586d96e427756
  nonce: 1849
  difficulty: 4

settle after confirmation
  HTTP 201
  block: 3
  block_hash: 000024325f90e44bd629de3ae9214952528ab87c0ac68d9bbdc586d96e427756
  holder: Lindiwe
  mzar_burned: 75000
  paid_at: 1791705840
  rand_paid: 75000
  receipt_hash: aaa3d5576df545eedee0f247a12296c9eb701dae325c3ab7e9e0120075be615b
  receipt_no: RCPT-0001
  redeem_tx: 5270433f2cc0072f76c63d8e817b88364779b0c08891936b93fe63fd44e0e67a



,mZAR on chain,ZAR in bank
Lindiwe,0.00 mZAR,R750.00
RESERVE,0.00 mZAR,R250.00
Sipho,250.00 mZAR,"R4,000.00"


reserve: 250.00 mZAR in circulation, R250.00 held, ratio 1.0, fully backed: True, balances add up: True, identity holds: True, mints match deposits: True


### Step v. Verification and explorer

First the server audits the whole chain. Then we do something more important: we check the transfer **without trusting the server's answer**. The function below takes only the public chain data and re-does every check itself: signature, id, Merkle proof, block hash, proof of work, and the links of every later block back to the one that holds the payment.

`independent_check` is a *full-node* check: it downloads every transaction in the block. A **light client** needs far less. `light_client_check` below downloads only the block *headers* and asks the server for a short Merkle proof, then verifies that proof itself against the header's root and checks the header hash, the proof of work and the links to the tip.

In [15]:
status, audit = call("GET", "/chain/verify")
show("full chain audit", status, audit)
evidence["audit"] = audit

def independent_check(public_chain, tx_id, difficulty):
    """Re-do every check using only public chain data."""
    for block in public_chain:
        ids = [t["id"] for t in block["transactions"]]
        if tx_id not in ids:
            continue
        pos = ids.index(tx_id)
        tx = block["transactions"][pos]
        links_ok = all(public_chain[i]["previous_hash"] == public_chain[i - 1]["hash"]
                       for i in range(block["index"] + 1, len(public_chain)))
        checks = {
            "signature is valid": verify_signature(tx["sender"], canonical(tx_body(tx)), tx["signature"]),
            "id matches the signed contents": tx["id"] == sha256(canonical(tx_body(tx))),
            "merkle proof reaches the block root": verify_proof(tx_id, merkle_proof(ids, pos), block["merkle_root"]),
            "block hash matches its header": block["hash"] == header_hash(block),
            "block meets the proof-of-work target": block["hash"].startswith("0" * difficulty),
            "later blocks link back to it": links_ok,
        }
        return block["index"], checks
    return None, {"transaction found in chain": False}

_, public = call("GET", "/chain")
where, checks = independent_check(public["chain"], transfer_tx["id"], DIFFICULTY)
print(f"transfer {transfer_tx['id'][:16]}... found in block {where}")
for name, ok in checks.items():
    print(f"  [{'pass' if ok else 'FAIL'}] {name}")
evidence["independent_check"] = checks

status, info = call("GET", f"/transactions/{transfer_tx['id']}")
print("\nserver lookup:", info["status"], "| block", info["block"], "| confirmations", info["confirmations"],
      "| proof steps", len(info["merkle_proof"]), "| proof valid", info["proof_valid"])
evidence["lookup"] = {k: info[k] for k in ("status", "block", "block_hash", "confirmations", "merkle_root", "proof_valid")}

def light_client_check(headers, info, tx_id, difficulty):
    """Headers only, plus the server's Merkle proof. No transactions are downloaded."""
    h = headers[info["block"]]
    return {
        "header hash matches": h["hash"] == header_hash(h),
        "header meets the proof-of-work target": h["hash"].startswith("0" * difficulty),
        "server's proof reaches the header's merkle root": verify_proof(tx_id, info["merkle_proof"], h["merkle_root"]),
        "header links to the one before it": info["block"] == 0 or h["previous_hash"] == headers[info["block"] - 1]["hash"],
        "later headers link back to it": all(headers[i]["previous_hash"] == headers[i - 1]["hash"]
                                            for i in range(info["block"] + 1, len(headers))),
    }

headers = [{k: v for k, v in b.items() if k != "transactions"} for b in public["chain"]]
light = light_client_check(headers, info, transfer_tx["id"], DIFFICULTY)
print("\nlight-client check (headers + Merkle proof only):")
for name, ok in light.items():
    print(f"  [{'pass' if ok else 'FAIL'}] {name}")
evidence["light_client_check"] = light

full chain audit
  HTTP 200
  blocks: 4
  message: chain is valid (4 blocks, supply 250.00 mZAR)
  valid: True

transfer 49ecdae540d323ef... found in block 2
  [pass] signature is valid
  [pass] id matches the signed contents
  [pass] merkle proof reaches the block root
  [pass] block hash matches its header
  [pass] block meets the proof-of-work target
  [pass] later blocks link back to it

server lookup: confirmed | block 2 | confirmations 2 | proof steps 0 | proof valid True

light-client check (headers + Merkle proof only):
  [pass] header hash matches
  [pass] header meets the proof-of-work target
  [pass] server's proof reaches the header's merkle root
  [pass] header links to the one before it
  [pass] later headers link back to it


## 10. Explorer

Two views of the same data. The text version is handy in a terminal. The HTML version is what `GET /explorer` serves, shown here inside the notebook.

In [16]:
def chain_text(service):
    lines = []
    for b in service.chain.chain:
        lines.append(f"Block #{b['index']}  hash {b['hash'][:20]}...  nonce {b['nonce']}  "
                     f"{len(b['transactions'])} tx  prev {b['previous_hash'][:10]}...")
        for t in b["transactions"]:
            lines.append(f"    {t['kind']:<8} {label(service, t['sender']):<8} -> {label(service, t['recipient']):<8} "
                         f"{t['amount'] / 100:>9,.2f} mZAR   id {t['id']}")
    return "\n".join(lines)

def explain(service, tx_id):
    info = service.chain.lookup(tx_id)
    if not info:
        print("unknown transaction")
        return
    tx = info["tx"]
    print(f"{tx['kind']} {tx['id']}")
    print(f"  from      {label(service, tx['sender'])}")
    print(f"  to        {label(service, tx['recipient'])}")
    print(f"  amount    {mzar(tx['amount'])}")
    if info["status"] == "pending":
        print("  status    pending, not in a block yet")
        return
    print(f"  status    confirmed in block #{info['block']} ({info['confirmations']} confirmation(s))")
    print(f"  block     {info['block_hash']}")
    print(f"  merkle    proof of {len(info['merkle_proof'])} step(s) is {'valid' if info['proof_valid'] else 'INVALID'}")

print(chain_text(service))
print()
explain(service, transfer_tx["id"])
evidence["explorer_text"] = chain_text(service)

Block #0  hash 0000adebcfbaf1d8fc50...  nonce 20941  0 tx  prev 0000000000...
Block #1  hash 000063e3b38528241200...  nonce 7663  1 tx  prev 0000adebcf...
    MINT     Reserve  -> Sipho     1,000.00 mZAR   id 9f4b12c94b30b0db142baa799a0b79c76b1034504ebbd777ae256e98e6618491
Block #2  hash 00003bde11490d5d0f83...  nonce 59849  1 tx  prev 000063e3b3...
    TRANSFER Sipho    -> Lindiwe     750.00 mZAR   id 49ecdae540d323efd9e724a2925903a167f10d5ff49a171b52cfd708193a57d7
Block #3  hash 000024325f90e44bd629...  nonce 1849  1 tx  prev 00003bde11...
    REDEEM   Lindiwe  -> Reserve     750.00 mZAR   id 5270433f2cc0072f76c63d8e817b88364779b0c08891936b93fe63fd44e0e67a

TRANSFER 49ecdae540d323efd9e724a2925903a167f10d5ff49a171b52cfd708193a57d7
  from      Sipho
  to        Lindiwe
  amount    750.00 mZAR
  status    confirmed in block #2 (2 confirmation(s))
  block     00003bde11490d5d0f839a139948070d3b2e4ed06e86b921adf7d1b885101a32
  merkle    proof of 0 step(s) is valid


In [17]:
from IPython.display import HTML
page = requests.get(f"{BASE}/explorer", params={"tx": transfer_tx["id"]}).text
HTML(page)

Transaction id,Kind,From,To,mZAR
5270433f2cc0072f76c63d8e817b88364779b0c08891936b93fe63fd44e0e67a,REDEEM,Lindiwe,Reserve,750.00
Transaction id,Kind,From,To,mZAR
49ecdae540d323efd9e724a2925903a167f10d5ff49a171b52cfd708193a57d7 ← this payment,TRANSFER,Sipho,Lindiwe,750.00
Transaction id,Kind,From,To,mZAR
9f4b12c94b30b0db142baa799a0b79c76b1034504ebbd777ae256e98e6618491,MINT,Reserve,Sipho,"1,000.00"


### Step vi. Invalid transfers are rejected

Eleven attempts that should all fail. After step iv Sipho has 250.00 mZAR and his next nonce is 1.

*Cryptography and ledger rules*
1. **Bad signature.** Sign a payment of 100, then change the amount to 900 before sending.
2. **Insufficient balance.** A correctly signed payment of 900 when he holds 250.
3. **Replay.** Re-send the transfer that was already confirmed.
4. **Unauthorised mint.** Sipho tries to print himself 5,000 mZAR.
5. **Double payout.** Ask the reserve to settle Lindiwe's redemption a second time.

*Input validation*
6. **Fractional cents.** A payment of 12.345.
7. **Not a number.** An on-ramp of `"NaN"`.
8. **Wrong types.** A transaction whose nonce is the string `"0"`.

*Reserve integrity*
9. **Reserve-signed mint with no deposit.** Even the reserve's own valid signature is refused on the public endpoint, because every mint must be backed by a deposit.
10. **Name collision.** Creating a wallet called `RESERVE`, which would let its owner on-ramp from the reserve's own account.
11. **Redeem from an unregistered address.** It would burn coins that could never be paid out.

Nothing here reaches the pending pool, and the chain stays valid.

In [18]:
sipho = service.wallets["Sipho"]
lindiwe = service.wallets["Lindiwe"]
nonce = chain.next_nonce(sipho.address)
failures = []

# 1 bad signature
forged = make_tx(sipho, "TRANSFER", lindiwe.address, to_cents(100), nonce)
forged["amount"] = to_cents(900)
failures.append(("1 bad signature (amount changed after signing)", call("POST", "/transactions/new", **forged)))

# 2 insufficient balance
big = make_tx(sipho, "TRANSFER", lindiwe.address, to_cents(900), nonce)
failures.append(("2 insufficient balance (900 requested, 250 held)", call("POST", "/transactions/new", **big)))

# 3 replay of a confirmed transfer
failures.append(("3 replay of an already confirmed transfer", call("POST", "/transactions/new", **transfer_tx)))

# 4 unauthorised mint
fake_mint = make_tx(sipho, "MINT", sipho.address, to_cents(5000), nonce)
failures.append(("4 unauthorised mint by a normal user", call("POST", "/transactions/new", **fake_mint)))

# 5 double payout
failures.append(("5 second payout for the same redemption", call("POST", "/offramp/settle", tx_id=redeem_tx["id"])))

# 6 fractional cents
failures.append(("6 fractional cents (12.345)", call("POST", "/payments/send", **{"from": "Sipho", "to": "Lindiwe", "amount": "12.345"})))

# 7 not a number
failures.append(("7 on-ramp amount is NaN", call("POST", "/onramp", name="Sipho", amount_zar="NaN")))

# 8 wrong field types
wrong_types = {**make_tx(sipho, "TRANSFER", lindiwe.address, to_cents(10), nonce), "nonce": "0"}
failures.append(("8 wrong field type (nonce is a string)", call("POST", "/transactions/new", **wrong_types)))

# 9 reserve-signed mint with no deposit behind it
unbacked = make_tx(reserve_wallet, "MINT", sipho.address, to_cents(5000), chain.next_nonce(reserve_wallet.address))
failures.append(("9 reserve-signed mint without a deposit", call("POST", "/transactions/new", **unbacked)))

# 10 wallet named RESERVE
failures.append(("10 wallet called RESERVE", call("POST", "/wallets/new", name="RESERVE")))

# 11 redeem from an address with no bank account
stranger = demo_wallet("Stranger")
orphan = make_tx(stranger, "REDEEM", reserve_wallet.address, to_cents(100), 0)
failures.append(("11 redeem from an unregistered address", call("POST", "/transactions/new", **orphan)))

for title, (status, data) in failures:
    print(f"{title}\n  HTTP {status}  ->  {data.get('error')}")
evidence["failures"] = [{"case": t, "http": s, "error": d.get("error")} for t, (s, d) in failures]

print("\npending pool after all attempts:", len(chain.pending), "transactions")
status, audit = call("GET", "/chain/verify")
print("audit:", audit["message"])
evidence["reserve_final"] = reserve_line()

1 bad signature (amount changed after signing)
  HTTP 400  ->  invalid signature
2 insufficient balance (900 requested, 250 held)
  HTTP 400  ->  insufficient balance
3 replay of an already confirmed transfer
  HTTP 400  ->  bad nonce (expected 1, got 0)
4 unauthorised mint by a normal user
  HTTP 400  ->  only the reserve agent can mint
5 second payout for the same redemption
  HTTP 400  ->  this redemption was already paid out
6 fractional cents (12.345)
  HTTP 400  ->  amount: amount must have no more than two decimal places
7 on-ramp amount is NaN
  HTTP 400  ->  amount_zar: amount must be a positive finite number
8 wrong field type (nonce is a string)
  HTTP 400  ->  nonce must be a non-negative integer
9 reserve-signed mint without a deposit
  HTTP 400  ->  mints are created only by the on-ramp, which takes the rand first
10 wallet called RESERVE
  HTTP 400  ->  the name RESERVE is reserved for the reserve account
11 redeem from an unregistered address
  HTTP 400  ->  redeem send

## 11. Extra tests

### Tampering with history

Someone edits an old block, for example changing Sipho's payment from 750 to 7,500 inside a stored copy of the chain. We work on a deep copy so the live demo stays untouched.

In [19]:
tampered = copy.deepcopy(chain)
tampered.chain[2]["transactions"][0]["amount"] = to_cents(7500)
ok, message = tampered.verify_chain()
print("edit a transaction amount     ->", ok, "|", message)

# a smarter attacker also fixes the id and re-signs? They cannot, they do not hold Sipho's key.
tampered2 = copy.deepcopy(chain)
tampered2.chain[1]["nonce"] += 1
ok2, message2 = tampered2.verify_chain()
print("change a block header         ->", ok2, "|", message2)

evidence["tamper"] = [{"edit": "amount in block 2", "valid": ok, "message": message},
                      {"edit": "nonce in block 1", "valid": ok2, "message": message2}]

edit a transaction amount     -> False | block 2, tx 49ecdae540: invalid signature
change a block header         -> False | block 1: stored hash does not match the header


### A Merkle proof in a busier block

Each demo block holds one transaction, so its Merkle proof has no steps. To see a real proof we mine a block with four transactions on a **copy** of the chain and prove that one of them is inside, using only a short path of hashes.

In [20]:
busy = copy.deepcopy(chain)
sip, lin = service.wallets["Sipho"], service.wallets["Lindiwe"]
for amount in (10, 20, 30, 40):
    ok, why = busy.submit(make_tx(sip, "TRANSFER", lin.address, to_cents(amount), busy.next_nonce(sip.address)))
    assert ok, why
busy_block = busy.mine()
middle = busy_block["transactions"][2]
info = busy.lookup(middle["id"])
print(f"block {busy_block['index']} holds {len(busy_block['transactions'])} transactions")
print(f"proof for transaction #2 has {len(info['merkle_proof'])} steps:")
for sibling, side in info["merkle_proof"]:
    print(f"  sibling {sibling[:16]}... on the {side}")
print("proof valid:", info["proof_valid"], "| wrong id rejected:",
      not verify_proof(busy_block["transactions"][1]["id"], info["merkle_proof"], info["merkle_root"]))
evidence["busy_block"] = {"transactions": len(busy_block["transactions"]), "proof_steps": len(info["merkle_proof"]),
                          "proof_valid": info["proof_valid"], "merkle_root": info["merkle_root"]}

block 4 holds 4 transactions
proof for transaction #2 has 2 steps:
  sibling 247e7f0325b67a4e... on the right
  sibling 186ebe85ba766a60... on the left
proof valid: True | wrong id rejected: True


### How hard is the proof of work?

With $d$ leading hex zeros the success chance per attempt is $16^{-d}$, so the expected number of attempts is

$$E[N_d] = 16^{d}.$$

We mine ten headers at each difficulty and compare the measured mean with that formula. The number of attempts is geometric, so its standard deviation is about as large as its mean: with so few samples the mean can be off by tens of percent, which is normal and is why the measured column does not match $16^d$ exactly. Attempt counts are the same on every run (the headers are fixed). Seconds depend on your computer.

In [21]:
def time_pow(d, trials=10):
    rows = []
    for t in range(trials):
        block = {"index": t + 1, "timestamp": 0, "previous_hash": sha256(f"seed{t}"),
                 "merkle_root": sha256(f"root{d}-{t}"), "difficulty": d, "nonce": 0}
        start = time.perf_counter()
        while not header_hash(block).startswith("0" * d):
            block["nonce"] += 1
        rows.append((block["nonce"] + 1, time.perf_counter() - start))
    return rows

table = []
for d in range(1, 6):
    rows = time_pow(d)
    table.append({"difficulty d": d, "expected 16^d": 16 ** d,
                  "mean attempts": round(sum(r[0] for r in rows) / len(rows)),
                  "mean seconds": round(sum(r[1] for r in rows) / len(rows), 4)})
pow_df = pd.DataFrame(table)
display(pow_df)

fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.plot(pow_df["difficulty d"], pow_df["mean seconds"], "ko-", label="measured")
base = pow_df["mean seconds"].iloc[2] / 16 ** 3
ax.plot(pow_df["difficulty d"], [base * 16 ** d for d in pow_df["difficulty d"]], "--", color="0.5", label="16^d trend")
ax.set_yscale("log")
ax.set_xlabel("difficulty d (leading hex zeros)")
ax.set_ylabel("mean seconds per block")
ax.set_xticks(range(1, 6))
ax.grid(True, color="0.85")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("pow_timing.png", dpi=200)
plt.close(fig)
evidence["pow_table"] = table
print("saved pow_timing.png")

,difficulty d,expected 16^d,mean attempts,mean seconds
0,1,16,15,0.0000
1,2,256,397,0.0010
2,3,4096,3215,0.0058
3,4,65536,45266,0.0698
4,5,1048576,1125246,1.8028


saved pow_timing.png


### Mining while a payment arrives

The API server is threaded, so a customer can submit a payment while a block is being mined. In an earlier draft the pool was cleared *after* mining, which silently deleted any payment that arrived in between. The test below starts mining on a copy of the chain and submits a payment at the same moment. The lock makes the payment wait, and nothing is lost.

In [22]:
race = copy.deepcopy(chain)
sip, lin = service.wallets["Sipho"], service.wallets["Lindiwe"]
first = make_tx(sip, "TRANSFER", lin.address, to_cents(5), race.next_nonce(sip.address))
assert race.submit(first)[0]

def late_payment():
    tx = make_tx(sip, "TRANSFER", lin.address, to_cents(7), race.next_nonce(sip.address))
    late_payment.result = (tx, race.submit(tx))

miner = threading.Thread(target=race.mine)
payer = threading.Thread(target=late_payment)
miner.start(); payer.start()
miner.join(); payer.join()

late_tx, (late_ok, late_why) = late_payment.result
in_chain = late_tx["id"] in race.tx_location
in_pool = any(t["id"] == late_tx["id"] for t in race.pending)
print("first payment mined      :", first["id"] in race.tx_location)
print("late payment accepted    :", late_ok)
print("late payment kept        :", in_chain or in_pool, "(in the block)" if in_chain else "(in the pending pool)")
print("payments lost            :", 0 if (late_ok and (in_chain or in_pool)) else 1)
evidence["race_test"] = {"first_mined": first["id"] in race.tx_location, "late_accepted": late_ok,
                         "late_kept": in_chain or in_pool}

first payment mined      : True
late payment accepted    : True
late payment kept        : True (in the pending pool)
payments lost            : 0


### Non-custodial redemption

In the main demo the platform holds the demo wallets' keys, which is only acceptable for a demo. Here a separate, small platform shows the realistic path: **Thandi keeps her own key**, registers only her public address, signs her own REDEEM, and the reserve pays her after confirmation. The platform never sees her private key and cannot sign for her.

In [23]:
r2 = demo_wallet("Reserve2")
t_chain = Blockchain(r2.address, difficulty=2, clock=DemoClock())
t_service = PaymentService(t_chain, r2, Bank(), make_wallet=demo_wallet)

thandi = demo_wallet("Thandi")                       # on her own device; the service never receives this object
t_service.register_external("Thandi", thandi.address)
t_service.bank.accounts["Thandi"] = to_cents(300)    # her simulated bank balance

t_service.onramp("Thandi", to_cents(300)); t_chain.mine()
print("platform signing for Thandi:", end=" ")
try:
    t_service.send("Thandi", "Thandi", to_cents(1))
except PaymentError as e:
    print("refused ->", e)

redeem = make_tx(thandi, "REDEEM", r2.address, to_cents(200), t_chain.next_nonce(thandi.address))
ok, why = t_service.submit_external(redeem)          # signed on her device, submitted through the public door
print("redeem submitted by Thandi:", ok, "|", why)
t_chain.mine()
receipt2 = t_service.settle(redeem["id"])
status2 = t_service.reserve_status()
print("receipt:", receipt2["receipt_no"], "| paid", rand(receipt2["rand_paid"]), "to", receipt2["holder"])
print("Thandi bank:", rand(t_service.bank.accounts["Thandi"]), "| reserve bank:", rand(t_service.bank.accounts["RESERVE"]),
      "| supply:", mzar(status2["mzar_in_circulation"]))
print("reserve identity holds:", status2["reserve_identity_holds"], "| mints match deposits:", status2["mints_match_deposits"])
print("chain audit:", t_chain.verify_chain()[1])
assert status2["reserve_identity_holds"] and status2["mints_match_deposits"] and t_chain.verify_chain()[0]
evidence["noncustodial"] = {"redeem_tx": redeem["id"], "receipt": receipt2["receipt_no"],
                            "rand_paid": receipt2["rand_paid"], "identity_holds": status2["reserve_identity_holds"]}

platform signing for Thandi: refused -> Thandi is a non-custodial wallet: sign on your own device and POST the transaction to /transactions/new
redeem submitted by Thandi: True | accepted into the pending pool
receipt: RCPT-0001 | paid R200.00 to Thandi
Thandi bank: R200.00 | reserve bank: R100.00 | supply: 100.00 mZAR
reserve identity holds: True | mints match deposits: True
chain audit: chain is valid (3 blocks, supply 100.00 mZAR)


## 12. Save the evidence and shut the server down

The report quotes transaction ids, block hashes and outputs straight from `evidence.json`, so the numbers in the PDF are the numbers from this run.

## 13. What this prototype does not do

- **One operator.** One process mines, validates and runs the reserve. A public chain would have many independent miners or validators.
- **No fees, no miner reward, no incentive design.** Proof of work here only shows the mechanism.
- **Single currency, fixed 1:1 rule.** There is no exchange rate, no FX spread and no second currency. The rule is realistic only for rand-pegged partners.
- **Simulated bank and reserve.** No real money, no audit of the reserve, no segregation of client funds.
- **Custodial demo keys.** The service signs for demo wallets and keys come from names. Real wallets keep keys on the user's device (Mastering Bitcoin, Chapters 4 and 5).
- **No KYC, AML or sanctions screening.** Any address can be on-ramped. This is the largest gap against a licensed product and it gets its own section in the report.
- **No networking.** The `/nodes` endpoints store a peer list but there is no gossip or fork choice (van Flymen Chapter 5 shows how to add them).
- **No privacy.** Every amount and address is visible on the chain.
- **No authentication on the custodial endpoints.** `/onramp`, `/payments/send` and `/offramp` act on a wallet *by name*, so anyone who can reach the server can move that wallet's money. A real service would authenticate every caller and check that the caller owns the wallet.
- **Demo Reserve key is guessable.** It is derived from the name "Reserve". The API refuses any MINT on the public endpoint, but anyone who can recompute that key could still sign a valid one off-chain. Use `MZAR_RESERVE_SEED` outside a demo.
- **Backing is checked off-chain.** The ledger cannot see the bank, so `verify_chain` proves the chain is internally consistent, not that the reserve holds the rand. `/reserve` checks that, but only the reserve agent runs it.
- **No durable storage.** Everything is in memory and is lost when the kernel stops.
- **Settlement is not truly atomic.** The code marks a redemption as paid before moving rand and undoes that on failure, but a crash between two real systems (a bank and a chain) needs a proper recovery process.

In [24]:
with open("evidence.json", "w") as f:
    json.dump(evidence, f, indent=2)
server.stop()
print("evidence.json written, server stopped")

evidence.json written, server stopped
